# Errors in code prompts

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches

from gsm_benchmarker.results_analyser import MultiVariantMultiModelResultsAnalyser


plt.style.use('seaborn-v0_8-muted')
plt.style.use('seaborn-v0_8-darkgrid')


In [ ]:
pp = Path("../../../../data/gsm-symbolic/outputs").resolve()
p_code_short = pp / 'code_full__09_02/corrected'  # TODO: update
p_code_long = pp / 'code_no_sep_full__12_03/final'

In [ ]:
mres_code_short = MultiVariantMultiModelResultsAnalyser(p_code_short)
mres_code_long = MultiVariantMultiModelResultsAnalyser(p_code_long)

In [ ]:
models = [
    'Mathstral-7B-v0.1',
    'Phi-3-mini-128k-instruct',
    'Phi-3.5-mini-instruct',
    'gemma-2-9b'
]

In [ ]:
ERROR_COLORS = {
    'NAME_ERROR': 'tab:blue',
    'TYPE_VALUE_ERROR': 'tab:green',
    'SYNTAX_ERROR': 'tab:red',
    'NONE_RETURNED': 'tab:orange',
    'NOT_A_NUMBER': 'tab:purple',
    'Other': '#ca9161'              # Brown (for any catch-all category)
}

### Error types seen in code output prompt experiments

In [ ]:
mres_code_long.variants['main'].full_data.groupby('model').size().iloc[0]

# TODO
- fix colours - different errors have different ones (syntax is pink / red etc)
- remove axis titles where not needed
- counts on left axis, percentages on right - check if correct
- plot error _models_ by question id - see if the same questions were difficult for different models
- make a table for Latex
- combine per-model bar charts into one figure


In [ ]:
_ = mres_code_long.variants['main'].plot_error_types_by_model(models=models, stacked=False, bar_labels=True)


In [ ]:
# all questions
_ = mres_code_long.variants['main'].plot_error_types_by_question_id(models=models)

In [ ]:
mres_code_long.variants['main'].get_failed_answer_cases(models=models)

In [ ]:
# 25 most difficult questions
_ = mres_code_long.variants['main'].plot_error_types_by_question_id(max_questions=25, models=models, title="Error types - all models")

In [ ]:
mres_code_long.variants['main'].get_percentage_total('id')

In [ ]:
mres_code_long.variants['main'].get_percentage_total('model')


In [ ]:
mres_code_long.variants['main'].models

In [ ]:
models

In [ ]:
for model in models:
    if model in mres_code_long.variants['main'].models:
        _ = mres_code_long.variants['main'].plot_error_types_by_question_id(max_questions=25, models=model, title=f"Error types - {model}")


In [ ]:
df_long = mres_code_long.variants['main'].get_failed_answer_cases(models=models)
df_short = mres_code_short.variants['main'].get_failed_answer_cases(models=models)

In [ ]:
df_long.columns

In [ ]:
df_long.model.unique()

In [ ]:
df_short.model.unique()

In [ ]:
import pandas as pd

df_long['prompt'] = len(df_long) * ['long']
df_short['prompt'] = len(df_short) * ['short']

df = pd.concat([df_long, df_short], ignore_index=True)
df = df[['model', 'id', 'instance', 'question', 'detected_result_pattern', 'full_response', 'prompt']]
df = df.rename(columns={'detected_result_pattern': 'error_type'})

In [ ]:
df.columns

In [ ]:
df.error_type.unique()

In [ ]:
grouped = df.groupby(['prompt', 'model', 'error_type']).size().unstack(fill_value=0)
grouped = grouped.reindex(columns=ERROR_COLORS.keys(), fill_value=0)

grouped_pct = grouped.div(grouped.sum(axis=1), axis=0) * 100

prompts = df.prompt.unique().tolist()


other_colour_used = False

def get_colours(_data):
    global other_colour_used
    c = [ERROR_COLORS.get(error_type, ERROR_COLORS['Other']) for error_type in _data.columns]
    if _data['Other'].any():
       other_colour_used = True
    return c


fig, axes = plt.subplots(len(prompts), 2, figsize=(10, 6), sharey='row', sharex='col')

bar_kwargs = dict(edgecolor='white', linewidth=0.5, kind='barh', stacked=True, legend=False)

for i, prompt_type in enumerate(prompts):

    # Extract data for the specific prompt
    count_data = grouped.loc[prompt_type]
    pct_data = grouped_pct.loc[prompt_type]

    # Plot stacked bar charts
    count_data.plot(ax=axes[i, 0], color=get_colours(pct_data), **bar_kwargs)
    pct_data.plot(ax=axes[i, 1], color=get_colours(pct_data), **bar_kwargs)

    # Use the Y-axis label to indicate the prompt type for that entire row
    axes[i, 0].set_ylabel(f"{prompt_type.capitalize()} code prompt")

    # Format X-axis for percentages (Col 1)
    axes[i, 1].xaxis.set_major_formatter(mtick.PercentFormatter())

    total_counts = count_data.sum(axis=1)
    x_offset = total_counts.max() * 0.02

    for y_position, total in enumerate(total_counts):
        if total > 0:
            # Annotate only on the count plots (Col 0)
            axes[i, 0].text(x=total + x_offset, y=y_position, s=f"{int(total)}",
                            ha='left', va='center', fontsize=9)

    # Add padding to the right of the count x-axis so the labels aren't cut off
    axes[i, 0].margins(x=0.15)


axes[0, 0].set_title("Error Count", pad=15)
axes[0, 1].set_title("Proportion of Total Errors", pad=15)
axes[1, 0].set_xlabel("Error Count")
axes[1, 1].set_xlabel("Proportion of Total Errors")

# Add a single shared legend at the bottom
legend_patches = [mpatches.Patch(color=color, label=error_name.replace('_', ' ')) for error_name, color in ERROR_COLORS.items()]
if not other_colour_used:
    legend_patches = legend_patches[:-1]

fig.legend(handles=legend_patches, loc='lower center', ncol=len(legend_patches),
           title="Error Type", frameon=True, framealpha=0.5, fontsize=8)

# Adjust layout to prevent overlap, leaving room at the bottom for the legend
fig.tight_layout()
fig.subplots_adjust(bottom=0.15)